# Step 1 — Can we actually find the eyes?

Before building anything on top of a face-tracking library, the first
question is the most boring one: **does it actually work on our footage?**
If MediaPipe can't reliably find a face and its eye landmarks, nothing
downstream (blink detection, gaze direction, ...) will be trustworthy
either.

This notebook:
- finds every recorded clip in this folder,
- runs Google's MediaPipe Face Mesh on every frame,
- draws the eye outline and iris centre on a few sample frames so you can
  *see* what the model is finding,
- and reports a detection rate for each clip.

No machine learning yet — this is just "does the sensor work."


## At a glance

| | |
|---|---|
| **Workshop step** | 1 of 6 |
| **Needs in this folder** | your recorded clips, named `blinks_<name>.mp4`, `eyes_only_<name>.mp4` and `head_only_<name>.mp4` (see `README.md`). In a group, put everyone's clips in the same folder. |
| **Produces** | nothing saved to disk. Prints a detection rate and shows four sample frames for each clip. |
| **Run time** | roughly 10 to 20 seconds per clip on a normal laptop |
| **Next** | `step_02_ear_blink_threshold.ipynb` |

**How to run:** in the menu choose *Run > Restart Kernel and Run All Cells*, or press Shift+Enter on each cell from the top down. Do not skip cells: later cells use variables made by earlier ones.

MediaPipe prints warnings in red or pink (libEGL, TensorFlow Lite, "feedback manager", protobuf). They are harmless, ignore them.

## What MediaPipe Face Mesh actually gives us

For every frame, if it finds a face, MediaPipe returns **468 landmark
points** (478 with iris refinement turned on) describing the face's shape.
Each landmark has:
- `x`, `y` — position, but **normalised to 0–1** (a fraction of the image
  width/height), not pixels
- `z` — rough relative depth (we won't use this)

We only care about a handful of these points: six around each eye (outer
corner, two upper-lid points, inner corner, two lower-lid points), plus the
iris centre and nose tip.

**Why convert to pixels?** A frame is rarely square. If you compute a
ratio (height/width) directly from normalised 0–1 coordinates, you're
silently distorting it by the frame's aspect ratio. Converting to pixels
first (`x * frame_width`, `y * frame_height`) keeps distances real.


In [ ]:
%matplotlib inline
import glob
import cv2
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt

# Every clip in this folder that follows the naming rule from README.md.
# <name> is whoever is in the clip, so a group of three has nine clips here.
CLIPS = sorted(glob.glob("blinks_*.mp4") + glob.glob("eyes_only_*.mp4") + glob.glob("head_only_*.mp4"))
if not CLIPS:
    raise SystemExit("No clips found. Save your recordings in this folder as blinks_<name>.mp4, "
                     "eyes_only_<name>.mp4 and head_only_<name>.mp4 (see README.md).")
print(f"Found {len(CLIPS)} clips:")
for clip in CLIPS:
    print("  ", clip)

# MediaPipe face mesh indices for the eye outline and iris centres.
# Order: outer corner, 2 upper-lid points, inner corner, 2 lower-lid points.
R_EYE = [33, 160, 158, 133, 153, 144]
L_EYE = [362, 385, 387, 263, 373, 380]
IRISES = [468, 473]

**Before running anything:** you recorded three kinds of clip. In one the
head stays still, in another the head turns. Which kind do you expect to
have the *lowest* detection rate, and why? What could make MediaPipe miss a
frame even with a cooperative, still subject?

In [ ]:
# ---------------------------------------------------------------------
# TRY IT: these two numbers (between 0 and 1) say how sure MediaPipe must
# be before it reports a face. Try 0.9 (strict) and 0.1 (lenient) and run
# the notebook again. Does the detection rate change? Do the dots on the
# sample frames get worse when it is lenient?
# ---------------------------------------------------------------------
MIN_DETECTION_CONFIDENCE = 0.5
MIN_TRACKING_CONFIDENCE = 0.5

# TRY IT: which frames to show, as fractions of the clip length.
# 0.1 = a frame 10% of the way through. Add or change values to look at
# moments you care about (for example a head turn or a blink).
SAMPLE_POSITIONS = (0.1, 0.4, 0.7, 0.95)


def check_clip(path):
    """Run face mesh over one clip.

    Returns (frames with a face, total frames, sample frames), where
    sample frames is a dict: frame index -> annotated RGB image.
    """
    mesh = mp.solutions.face_mesh.FaceMesh(
        max_num_faces=1,
        refine_landmarks=True,   # also gives us the iris landmarks (468, 473)
        min_detection_confidence=MIN_DETECTION_CONFIDENCE,
        min_tracking_confidence=MIN_TRACKING_CONFIDENCE,
    )
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise SystemExit(f"Could not open {path}. Is it a readable .mp4? (see README.md)")
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    sample_idx = set(int(n_frames * f) for f in SAMPLE_POSITIONS)

    total = detected = 0
    sample_frames = {}
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        total += 1
        h, w = frame.shape[:2]

        result = mesh.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if result.multi_face_landmarks:
            detected += 1
            lm = result.multi_face_landmarks[0].landmark
            # landmarks are fractions of the image (0-1): multiply by the
            # frame size to get pixel positions we can draw at
            for i in R_EYE + L_EYE:
                cv2.circle(frame, (int(lm[i].x * w), int(lm[i].y * h)), 2, (0, 255, 0), -1)
            for i in IRISES:
                cv2.circle(frame, (int(lm[i].x * w), int(lm[i].y * h)), 3, (0, 0, 255), -1)

        if total - 1 in sample_idx:
            sample_frames[total - 1] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    cap.release()
    mesh.close()
    return detected, total, sample_frames

## Running through every clip

We process every frame of every clip once. For speed we won't pop up a
live window for each one. Instead we keep a running count of "face found"
vs "face not found", and separately grab a few representative frames to
actually look at afterwards. (This is also more portable across laptops
than a live `cv2.imshow` window, which needs a working display.)

This is the slowest cell in the notebook: MediaPipe looks at every frame.

In [ ]:
samples_by_clip = {}
for clip in CLIPS:
    detected, total, samples_by_clip[clip] = check_clip(clip)
    rate = detected / total if total else 0.0
    print(f"{clip:<32} face detected in {detected}/{total} frames ({rate:.1%})")

Now let's actually look at what it found, on a few sample frames spread across each clip:

In [ ]:
for clip, sample_frames in samples_by_clip.items():
    if not sample_frames:
        print(f"{clip}: no sample frames to show")
        continue
    fig, axes = plt.subplots(1, len(sample_frames), figsize=(4 * len(sample_frames), 4), squeeze=False)
    for ax, (idx, img) in zip(axes[0], sorted(sample_frames.items())):
        ax.imshow(img)
        ax.set_title(f"frame {idx}")
        ax.axis("off")
    fig.suptitle(clip)
    plt.tight_layout()
    plt.show()

## Discuss

- Green dots trace the eye outline, the red dot is the iris centre. Do
  they look right to you on all four sample frames?
- If the detection rate wasn't 100%, which frames failed? (Adapt the loop
  above to save a frame when `result.multi_face_landmarks` is empty, then
  look at it — motion blur? face partly out of frame? lighting?)
- Compare the detection rates across your group's clips. If they differ,
  what could explain it (light, glasses, distance from the camera, how far
  the head turned)? A clip with a clearly lower rate is worth recording
  again before you build on it.
- This "does the sensor even work" check matters more than it looks:
  everything in the next notebooks silently assumes this landmark data is
  trustworthy.

**Next:** `step_02` turns six of these eye landmarks into a single number
per frame (the eye aspect ratio) and tries to detect blinks from it.
